In [170]:
#%pip install rapidfuzz

In [171]:
import pandas as pd
from rapidfuzz import process, fuzz
import numpy as np
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Sample df1 with words to match
df_ACFull = pd.read_csv(r"path/filename.csv")
df_ACFull

In [ ]:
# Sample df1 with words to match
df_FBGFull = pd.read_csv(r"path/filename.csv")
df_FBGFull

In [174]:
df_FBGFull=df_FBGFull[
    (df_FBGFull['Parent-Child- Customer Brand']=="Parent") & 
    (df_FBGFull['Attribute_Type']!="Autocare_Attribute")
    ].reset_index(drop=True)

In [175]:
df_FBGFull=df_FBGFull[['Product Group', 'Source', 'BrandName', 'Brand','PartTerminologyName', 'PAName','Parent-Child- Customer Brand']].drop_duplicates().reset_index(drop=True)

In [176]:
df_FBGFull.size

29722

In [177]:
PartNames=list(set(df_FBGFull['PartTerminologyName'].tolist()))

In [178]:
cols =['PartTerminologyName','Matching Autocare Attribute'] #,'Review_Mentions','Standards'
df_New = pd.DataFrame(columns=cols)
count=0

In [179]:

# Function to get approximate matches with a configurable threshold
def get_matches(word, reference_list, threshold=98):
    results = process.extract(word, reference_list, scorer=fuzz.ratio)
    return [match for match, score, _ in results if score >= threshold]

# Set your desired threshold here
MATCH_THRESHOLD = 45

In [180]:
for i in range (len(PartNames)):
    df_FBGF=df_FBGFull[df_FBGFull['PartTerminologyName']==PartNames[i]].reset_index(drop=True)
    df_ACF=df_ACFull[df_ACFull['PartTerminologyName']==PartNames[i]]

    for j in range(len(df_FBGF)):
        #print(PartNames[i],df_FBGF['Cleaned Attributes'][j])
        df_New.at[count,'PartTerminologyName']=PartNames[i]
        df_New.at[count,'FBG Attributes']=df_FBGF['PAName'][j]
        df_New.at[count,'Product Group']= df_FBGF['Product Group'][j]
        df_New.at[count,'Source']= df_FBGF['Source'][j]
        df_New.at[count,'BrandName']= df_FBGF['BrandName'][j]
        df_New.at[count,'Brand']= df_FBGF['Brand'][j]
        df_New.at[count,'Matching Autocare Attribute']=get_matches(df_FBGF['PAName'][j],df_ACF['PAName'].tolist(), threshold=MATCH_THRESHOLD)
        count=count+1

In [ ]:
df_New

In [ ]:
df_New = df_New[df_New['Matching Autocare Attribute'].apply(str) != "[]"].reset_index(drop=True)
df_New = df_New.drop_duplicates(subset=['PartTerminologyName', 'FBG Attributes']).reset_index(drop=True)
df_New

In [ ]:
df_New.to_excel(r'path/filename.xlsx', index=False)

## Misc Code


In [ ]:

# Sample df1 with words to match
df_FB = pd.read_excel(r"path/filename.xlsx",sheet_name="FB")
df_FB

In [ ]:
df_AC = pd.read_excel(r"path/filename.xlsx",sheet_name="AC")
df_AC

In [ ]:

# Function to get approximate matches with a configurable threshold
def get_matches(word, reference_list, threshold=98):
    results = process.extract(word, reference_list, scorer=fuzz.ratio)
    return [match for match, score, _ in results if score >= threshold]

# Set your desired threshold here
MATCH_THRESHOLD = 80


In [ ]:

# Apply the function to df1
df_FB['approx_matches'] = df_FB['Cleaned FBG Attribute'].apply(
    lambda w: get_matches(w, df_AC['AC Attribute'].tolist(), threshold=MATCH_THRESHOLD)
)

df_FB=df_FB.drop_duplicates(inplace=True).rest_index(drop=True)


In [ ]:
df_FB.to_excel(r'path/filename.xlsx', index=False)